In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

import yaml

config = yaml.safe_load(open("../config.yaml", "r"))

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"
import torch
from src.model.models_fno_2 import FNO_2
from src.model.ffno_layer import FactorizedSpectralConv3d
from src.dataloader.dataloader_3d import dataset_sr
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
model = FNO_2(in_channel=5,n_channels=20, n_residual_blocks=4, n_operator_blocks=5, modes=8, shifting_modes=5,apply_constraint=False).to(device)
pytorch_total_params = sum(p.numel() for p in model.parameters())
print(pytorch_total_params)
x = torch.randn(1, 5, 32, 32, 32).to(device)
output = model(x, 4)

In [ ]:
output.shape

In [ ]:
x = torch.randn(1,5, 32, 32, 32).to(device)
start = time.time()
iterations = 100
for op_blocks in [2, 3, 4, 5, 6]:
    model = FNO_2(
        in_channel=5,
        n_channels=20,
        n_residual_blocks=4,
        n_operator_blocks=op_blocks,
        modes=16,
        shifting_modes=10,
        apply_constraint=False,
    ).to(device)

    start = time.time()
    for i in range(iterations):
        with torch.no_grad():
            output = model(x, 4)
    print(op_blocks,(time.time()-start)/iterations)

In [ ]:
x = torch.randn(1, 5, 32, 32, 32).to(device)
for op_blocks in [2, 3, 4, 5, 6]:
    with torch.no_grad():
        model = FNO_1(in_channel=5, 
                      n_channels=64, 
                      n_residual_blocks=5, 
                      n_operator_blocks=op_blocks,
                      modes=16).to(device)
        start = time.time()
        out = model(x, 4)
        del model
        del out
    print(op_blocks, time.time() - start) 


In [ ]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
print(pytorch_total_params)

In [ ]:
import torch

# This must be power of 2 in all dimensions
x = torch.randn(1, 64, 64, device="cuda", dtype=torch.float16)

y = torch.fft.rfftn(x)
print(y.shape)


# Loading model and data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = CNN().to(device)
cnn.load_state_dict(torch.load('model/cnn_basic', weights_only=True))

In [ ]:
dataset = dataset_sr(max_samples=30)

In [ ]:
data = dataset[0]
lr_state = data[1]
print(lr_state.shape)


In [ ]:
output = model(torch.unsqueeze(lr_state,0),4)
print(output.shape)

In [ ]:
hr_state_test, lr_state_test_tensor = test_dataset[3]
print(hr_state_test.size())

# Loading a test input

In [ ]:
hr_state_test, lr_state_test_tensor = test_dataset[15]
hr_state_test, lr_state_test = hr_state_test.numpy(), lr_state_test_tensor.numpy()

sr_state_test = cnn(lr_state_test_tensor.to(device))
sr_state_test = sr_state_test.cpu().detach().numpy()

In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

# plot cuts at this z level
z_level = 2
for i, state in enumerate([lr_state_test, hr_state_test, sr_state_test]):
    axes[i,0].imshow(state[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,0].set_title("Density")

    axes[i,1].imshow(np.sqrt(state[1, :, :, z_level]**2 + state[2, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,1].set_title("Velocity")

    axes[i,2].imshow(state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,2].set_title("Pressure")

    # equal aspect ratio
    axes[i,0].set_aspect('equal', 'box')
    axes[i,1].set_aspect('equal', 'box')
    axes[i,2].set_aspect('equal', 'box')

In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random
random_list = random.sample(range(80), 10)

fig, axes = plt.subplots(10, 6, figsize=(10, 15))

# plot cuts at this z level
z_level = 32
z_level_reduced = z_level // 4

for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
for i, list_i in enumerate(random_list):
    hr_state_test, lr_state_test_tensor = dataset[list_i]
    hr_state_test, lr_state_test = hr_state_test.numpy(), lr_state_test_tensor.numpy()

    axes[i,0].imshow(lr_state_test[0, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,0].set_ylabel(f"{list_i}")
    axes[i,1].imshow(np.sqrt(lr_state_test[1, :, :, z_level_reduced]**2 + lr_state_test[2, :, :, z_level_reduced]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,2].imshow(lr_state_test[4, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    
    axes[i,3].imshow(hr_state_test[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,4].imshow(np.sqrt(hr_state_test[1, :, :, z_level]**2 + hr_state_test[2, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,5].imshow(hr_state_test[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

    if i == 0:
        axes[i,0].set_title("Density")
        axes[i,1].set_title("Velocity")
        axes[i,2].set_title("Pressure")
        axes[i,3].set_title("Density")
        axes[i,4].set_title("Velocity")
        axes[i,5].set_title("Pressure")

    # equal aspect ratio
    axes[i,0].set_aspect('equal', 'box')
    axes[i,1].set_aspect('equal', 'box')
    axes[i,2].set_aspect('equal', 'box')

Spectral convolution test

In [ ]:
from src.model.models_dsfno_3d import SpectralConv3d

In [ ]:
test_spectral_conv = SpectralConv3d(5, 5, 20, 20, 20)

In [ ]:
hr_state_test, lr_state_test_tensor = train_dataset[3]
lr_state_test = lr_state_test_tensor.numpy()
lr_state_test_tensor = torch.unsqueeze(lr_state_test_tensor, 0)
print(lr_state_test_tensor.size())
test_output, out_ft = test_spectral_conv(lr_state_test_tensor, 4)

out_ft = torch.squeeze(out_ft).detach().numpy()
test_output = torch.squeeze(test_output).detach().numpy()

In [ ]:
print(np.shape(test_output ) , np.shape(out_ft))

In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
z_level = 14
fig, ax = plt.subplots(1, 4, figsize = (12, 3))
ax[0].imshow(np.absolute(out_ft[0, :, :, 3]),  norm = LogNorm())
ax[1].imshow(test_output[3, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
ax[2].imshow(hr_state_test.numpy()[3, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
ax[3].imshow(lr_state_test[3, :, :, z_level // 4].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

In [ ]:
import importlib
from src.model.models_dsfno_3d import DSFNO

# After making changes to the file, run:
import src.model.models_dsfno_3d
importlib.reload(src.model.models_dsfno_3d)
from src.model.models_dsfno_3d import DSFNO

In [ ]:
DSFNO_test = DSFNO(in_channel=5)

In [ ]:
test_output = DSFNO_test(lr_state_test_tensor, 4)

In [ ]:
test_output.shape